# 3 Traducción automática neuronal

**Neural machine translation (NMT):** $\;$ aplicación original de Transformer

**AT25:** $\;$ AT25 se basa en PyTorch; fijamos la semilla para generación de números aleatorios

In [1]:
import torch; import torch.nn as nn; torch.manual_seed(23)
import import_ipynb; from at251 import create_model, create_causal_mask

**Modelo:** $\;$ modelo aleatorio simple

In [2]:
model = create_model(src_vocab_size=3, tgt_vocab_size=3, embed_dim=2, num_layers=1, num_heads=1, dropout=0.)

**Embedding del texto de entrada:** $\;$ secuencia de vectores de `embed_dim` dimensiones

In [3]:
src = torch.LongTensor([[1, 2, 1]]); model.src_embed(src).data

tensor([[[ 0.0415,  2.2411],
         [ 1.2927, -0.6469],
         [ 0.9508,  0.8249]]])

**Embedding del texto de salida:** $\;$ secuencia de vectores de `embed_dim` dimensiones

In [4]:
tgt = torch.LongTensor([[0]]); model.tgt_embed(tgt).data

tensor([[[1.3548, 1.4321]]])

**Proyección lineal final:** $\;$ `generator` transforma la salida del decoder en la predicción probabilística del modelo

In [5]:
model.generator.final_proj.weight = nn.Parameter(torch.randn_like(model.generator.final_proj.weight))
model.generator.final_proj(torch.tensor([[[-0.7071,  0.7071]]])).data

tensor([[[-0.0080, -0.3012, -0.0413]]])


<p style="page-break-after:always;"></p>


**Evaluación:**

In [6]:
model.eval() 
src = torch.LongTensor([[1, 2, 1]]); print(f"Texto de entrada: {src}")
src_mask = torch.ones(1, 1, 3); print(f"Máscara de entrada: {src_mask}\n")
memory = model.encode(src, src_mask)
ys = torch.zeros(1, 1).type_as(src)     # note: ys[0]=0, i.e., ys starts with 0
for i in range(2):
    print(f"Inferencia del siguiente token tras haber predicho {i} tokens:")
    print(f"* Texto de salida ya predicho: {ys}")
    tgt_mask = create_causal_mask(ys.size(1)).type_as(src.data)
    print(f"* Máscara de salida: {str(tgt_mask).replace('\n','')}")
    out = model.decode(ys, memory, src_mask, tgt_mask)
    print(f"* Salida del decoder: {str(out.data).replace('\n','')}")
    prob = model.generator(out[:, -1])     # last token
    print(f"* Logsoftmax del unembedding del último token de la salida: {str(prob.data).replace('\n','')}")
    _, next_word = torch.max(prob, dim=1)
    next_word = next_word.data[0]
    ys = torch.cat([ys, torch.empty(1, 1).type_as(src.data).fill_(next_word)], dim=1)
    print("* Texto de salida con nuevo token inferido:", ys, "\n")


Texto de entrada: tensor([[1, 2, 1]])
Máscara de entrada: tensor([[[1., 1., 1.]]])

Inferencia del siguiente token tras haber predicho 0 tokens:
* Texto de salida ya predicho: tensor([[0]])
* Máscara de salida: tensor([[[1]]])
* Salida del decoder: tensor([[[-0.7071,  0.7071]]])
* Logsoftmax del unembedding del último token de la salida: tensor([[-0.9981, -1.2913, -1.0314]])
* Texto de salida con nuevo token inferido: tensor([[0, 0]]) 

Inferencia del siguiente token tras haber predicho 1 tokens:
* Texto de salida ya predicho: tensor([[0, 0]])
* Máscara de salida: tensor([[[1, 0],         [1, 1]]])
* Salida del decoder: tensor([[[-0.7071,  0.7071],         [ 0.7071, -0.7071]]])
* Logsoftmax del unembedding del último token de la salida: tensor([[-1.2162, -0.9231, -1.1830]])
* Texto de salida con nuevo token inferido: tensor([[0, 0, 1]]) 




<p style="page-break-after:always;"></p>
